In [1]:
#Step 1: Import libraries
import pandas as pd
import numpy as np

In [7]:
# Step 2: Load the CSV file
df = pd.read_csv("extinct_animal.csv")

In [8]:
# Step 3: Inspect the data
df.head()

,Common Name,Scientific Name,Date of Extinction,Continent,Primary Reason for Extinction,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,Dodo,Raphus cucullatus,1662,Africa (Mauritius),Hunting & Invasive Species,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Stellera's Sea Cow,Hydrodamalis gigas,1768,Asia/North America,Overhunting (for meat),NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Great Auk,Pinguinus impennis,1844,Europe/North America,Overhunting (for feathers/meat),NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Quagga,Equus quagga quagga,1883,Africa,Overhunting & Competition with Livestock,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Passenger Pigeon,Ectopistes migratorius,1914,North America,Intensive Hunting & Habitat Loss,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.shape

(960, 25)

In [10]:
df.columns

Index(['Common Name', 'Scientific Name', 'Date of Extinction', 'Continent',
       'Primary Reason for Extinction', 'Unnamed: 5', 'Unnamed: 6',
       'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11',
       'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15',
       'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19',
       'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23',
       'Unnamed: 24'],
      dtype='object')

In [11]:
df.dtypes

Common Name                       object
Scientific Name                   object
Date of Extinction                object
Continent                         object
Primary Reason for Extinction     object
Unnamed: 5                       float64
Unnamed: 6                       float64
Unnamed: 7                       float64
Unnamed: 8                       float64
Unnamed: 9                       float64
Unnamed: 10                      float64
Unnamed: 11                      float64
Unnamed: 12                      float64
Unnamed: 13                      float64
Unnamed: 14                      float64
Unnamed: 15                      float64
Unnamed: 16                      float64
Unnamed: 17                      float64
Unnamed: 18                      float64
Unnamed: 19                      float64
Unnamed: 20                      float64
Unnamed: 21                      float64
Unnamed: 22                      float64
Unnamed: 23                      float64
Unnamed: 24     

In [12]:
# Step 4: Remove empty columns (columns with no data)
df = df.dropna(axis=1, how='all')

In [13]:
# Step 5: Remove empty rows (rows with no data)
df = df.dropna(how='all').reset_index(drop=True)

In [14]:
# Step 6: Remove repeated header rows (where the first column repeats the column name, e.g., "Common Name")
df = df[df["Common Name"] != "Common Name"].reset_index(drop=True)

In [15]:
# Step 7: Remove extra spaces from text columns
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

In [16]:
# Step 8: Rename columns for consistency (lowercase and easy-to-reference names)
df = df.rename(columns={
    "Common Name": "common_name",
    "Scientific Name": "scientific_name",
    "Date of Extinction": "date_of_extinction",
    "Continent": "continent",
    "Primary Reason for Extinction": "primary_reason"
})

In [17]:
# Step 9: Remove duplicates based on "common_name" (keep the first occurrence)
df["common_name_clean"] = df["common_name"].astype(str).str.strip().str.lower()
df = df.loc[~df["common_name_clean"].duplicated(keep="first")].copy()
df = df.drop(columns="common_name_clean").reset_index(drop=True)

In [18]:
# Step 10: Clean the Continent column
# Replace "N. America" to "North America", and "S. America" to "South America"
df["continent"] = df["continent"].replace({
    r"\bN\.\s*America\b": "North America",
    r"\bS\.\s*America\b": "South America"
}, regex=True)

In [19]:
# Remove country names inside parentheses (like "Africa (Mauritius)" -> "Africa")
df["continent"] = df["continent"].str.replace(r"\s*\(.*?\)", "", regex=True)
df["continent"] = df["continent"].str.strip().str.title()

In [20]:
# Step 11: Clean Date of Extinction
# Replace invalid values with NaN (e.g., "Uncertain", "Possibly extinct")
invalid_values = [
    "Uncertain",
    "Possibly extinct",
    "Functionally collapsing now"
]
df["date_of_extinction"] = df["date_of_extinction"].replace(invalid_values, np.nan)

In [21]:
# Convert the column to numeric, any non-integer values become NaN
df["date_of_extinction"] = pd.to_numeric(df["date_of_extinction"], errors="coerce")

In [22]:
# Step 12: Clean Primary Reason for Extinction
# Standardize text formatting (title case)
df["primary_reason"] = df["primary_reason"].astype(str).str.strip().str.title()

In [23]:
# Step 13: Final Check
df.head()

,common_name,scientific_name,date_of_extinction,continent,primary_reason
0,Dodo,Raphus cucullatus,1662.0,Africa,Hunting & Invasive Species
1,Stellera's Sea Cow,Hydrodamalis gigas,1768.0,Asia/North America,Overhunting (For Meat)
2,Great Auk,Pinguinus impennis,1844.0,Europe/North America,Overhunting (For Feathers/Meat)
3,Quagga,Equus quagga quagga,1883.0,Africa,Overhunting & Competition With Livestock
4,Passenger Pigeon,Ectopistes migratorius,1914.0,North America,Intensive Hunting & Habitat Loss


In [24]:
df.shape

(64, 5)

In [25]:
df.dtypes

common_name            object
scientific_name        object
date_of_extinction    float64
continent              object
primary_reason         object
dtype: object

In [26]:
df.isnull().sum()

common_name           0
scientific_name       0
date_of_extinction    4
continent             0
primary_reason        0
dtype: int64

In [27]:
# Step 14: Save the cleaned dataset to a new CSV file
df.to_csv("extinct_animal_cleaned.csv", index=False)